# Musical Bingo — Initial Analysis

This notebook is a starting point for analysing the musical bingo JSON generated from the PDF.

It covers:
- loading and validating the dataset
- basic statistics
- song and artist frequencies
- repeated songs
- sheet-level exploration
- preparation for comparing bingo sheets against randomly ordered music playlists

The dataset contains **150 bingo sheets**, with **15 songs per sheet**.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

JSON_PATH = Path("extract_chatgpt.json")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Source file: {data['source_file']}")
print(f"PDF pages: {data['total_pdf_pages']}")
print(f"Sheets: {data['total_sheets']}")
print(f"Songs per sheet: {data['songs_per_sheet']}")

## 1. Validate the dataset

In [ ]:
assert len(data['sheets']) == data['total_sheets']

assert all(
    len(sheet['songs']) == data['songs_per_sheet']
    for sheet in data['sheets']
)

assert all(
    'artist' in song and 'title' in song
    for sheet in data['sheets']
    for song in sheet['songs']
)

print('Validation successful.')
print(f"Total song entries: {sum(len(s['songs']) for s in data['sheets']):,}")

## 2. Convert the JSON into a convenient DataFrame

In [ ]:
rows = []

for sheet in data['sheets']:
    for position, song in enumerate(sheet['songs'], start=1):
        rows.append({
            'sheet_id': sheet['sheet_id'],
            'pdf_page': sheet['pdf_page'],
            'sheet_on_page': sheet['sheet_on_page'],
            'position': position,
            'artist': song['artist'],
            'title': song['title'],
            'song': f"{song['artist']} — {song['title']}"
        })

songs_df = pd.DataFrame(rows)
songs_df.head(10)

## 3. Basic statistics

In [ ]:
total_entries = len(songs_df)
unique_songs = songs_df['song'].nunique()
unique_artists = songs_df['artist'].nunique()

stats = pd.Series({
    'Bingo sheets': songs_df['sheet_id'].nunique(),
    'Songs per sheet': songs_df.groupby('sheet_id').size().iloc[0],
    'Total song entries': total_entries,
    'Unique songs': unique_songs,
    'Unique artists': unique_artists,
    'Average appearances per unique song': total_entries / unique_songs,
})

stats

## 4. Most frequently appearing songs

In [ ]:
song_frequency = (
    songs_df.groupby(['artist', 'title'], as_index=False)
    .size()
    .rename(columns={'size': 'appearances'})
    .sort_values('appearances', ascending=False)
)

song_frequency['song'] = song_frequency['artist'] + ' — ' + song_frequency['title']
song_frequency.head(30)

In [ ]:
top_n = 20

plot_df = song_frequency.head(top_n).sort_values('appearances')

plt.figure(figsize=(10, 8))
plt.barh(plot_df['song'], plot_df['appearances'])
plt.xlabel('Number of bingo sheets')
plt.title(f'Top {top_n} most frequent songs')
plt.tight_layout()
plt.show()

## 5. Most frequent artists

In [ ]:
artist_frequency = (
    songs_df.groupby('artist')
    .agg(
        appearances=('song', 'size'),
        unique_songs=('song', 'nunique')
    )
    .sort_values(['appearances', 'unique_songs'], ascending=False)
)

artist_frequency.head(30)

## 6. Repeated songs and distribution

In [ ]:
frequency_distribution = song_frequency['appearances'].value_counts().sort_index()

print('How many unique songs appear N times:')
display(frequency_distribution.rename('unique_songs'))

print('\nSongs appearing more than once:')
display(song_frequency[song_frequency['appearances'] > 1].reset_index(drop=True))

## 7. Inspect an individual bingo sheet

In [ ]:
SHEET_ID = 1

sheet_view = songs_df[songs_df['sheet_id'] == SHEET_ID].copy()
sheet_view[['position', 'artist', 'title']]

## 8. Compare overlap between bingo sheets

In [ ]:
sheet_sets = {
    sheet_id: set(group['song'])
    for sheet_id, group in songs_df.groupby('sheet_id')
}

def sheet_overlap(sheet_a, sheet_b):
    return len(sheet_sets[sheet_a] & sheet_sets[sheet_b])

# Example: overlap of sheet 1 with every other sheet
reference_sheet = 1
overlaps = pd.Series({
    sheet_id: sheet_overlap(reference_sheet, sheet_id)
    for sheet_id in sheet_sets
    if sheet_id != reference_sheet
}).sort_values(ascending=False)

overlaps.head(20)

## 9. Prepare for playlist / bingo-duration analysis

The next step can be to provide one or more randomly ordered playlists and simulate playback.

For each playlist we can determine, for every bingo sheet:
- the position at which each of its 15 songs is played
- the position of the 5th, 10th, 15th, etc. matched song
- the first position at which the sheet reaches a bingo condition
- how many songs have been played when the first/last sheet wins
- how many sheets are still incomplete after a given number of played songs

The exact definition of **bingo** (for example, any 5 songs, a row, a column, or a specific layout) should be encoded explicitly before running the simulation.

In [ ]:
# Example helper: return the playback position of each song belonging to a sheet.
# `playlist` should be a list of song strings in playback order, using the same
# 'Artist — Title' representation as songs_df['song'].

def sheet_match_positions(sheet_id, playlist):
    target_songs = sheet_sets[sheet_id]
    return {
        song: position
        for position, song in enumerate(playlist, start=1)
        if song in target_songs
    }

# Example usage:
# playlist = songs_df['song'].drop_duplicates().sample(frac=1, random_state=42).tolist()
# positions = sheet_match_positions(1, playlist)
# positions

## 10. Useful next analyses

Potential extensions:

1. Import several randomly sorted playlists.
2. Normalize song names to handle spelling/capitalization differences.
3. Match playlist entries to bingo songs.
4. Simulate the playback of each playlist.
5. Implement the actual bingo-card winning rules.
6. Calculate the distribution of game duration across playlists and sheets.
7. Identify songs that strongly affect the time-to-bingo.
8. Test whether the 150 cards provide sufficiently balanced coverage of the song pool.